## Imports

In [1]:
import os
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
from skfda import FDataGrid
from skfda.preprocessing.smoothing import BasisSmoother
from skfda.representation.basis import BSplineBasis

## Set current dir to project root dir

In [2]:
def find_project_root():
    """Walk up from CWD until we find the project root."""
    markers = [".git", "Makefile", "renv.lock", ".Rprofile"]
    path = Path.cwd()
    while path != path.parent:
        if any((path / m).exists() for m in markers):
            return path
        path = path.parent
    raise FileNotFoundError("Could not find project root")

os.chdir(find_project_root())

## Helper functions

### `format_elapsed(seconds)`

Formats an elapsed time in seconds into a human-readable string following the
[NIST Guide to the SI, Chapter 7](https://www.nist.gov/pml/special-publication-811/nist-guide-si-chapter-7-rules-and-style-conventions-expressing-values)
conventions for expressing values of quantities with units.

Examples: `41.3 s`, `1 min 41.3 s`, `1 h 20 min 10.6 s`

In [3]:
def format_elapsed(seconds):
    if seconds >= 3600:
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = seconds % 60
        return f"{h} h {m} min {s:.1f} s"
    elif seconds >= 60:
        m = int(seconds // 60)
        s = seconds % 60
        return f"{m} min {s:.1f} s"
    else:
        return f"{seconds:.1f} s"

## Variables

In [4]:
# Define the top-level directory where subject folders are
root_dir = "./ds006018_per_stimuli"

# Get all CSV files in any subfolder
all_csv_files = sorted(Path(root_dir).rglob("*.csv"))

print(all_csv_files[:3])
print(f"Total files: {len(all_csv_files)}")

[PosixPath('ds006018_per_stimuli/sub-001/task-auditoryoddball_Stimulus_S1.csv'), PosixPath('ds006018_per_stimuli/sub-001/task-auditoryoddball_Stimulus_S180.csv'), PosixPath('ds006018_per_stimuli/sub-001/task-auditoryoddball_Stimulus_S70.csv')]
Total files: 4746


## Main

In [5]:
# ── Timer start ───────────────────────────────────────────────────────────────────────
start = time.time()
# ──────────────────────────────────────────────────────────────────────────────────────

# Define the root directories
input_root  = "ds006018_per_stimuli"
output_root = "ds006018_functional"

for path in all_csv_files:
    # 1. Load the data
    eeg_df = pd.read_csv(path)

    # 2. Reshape to time x epoch matrix for channel F7
    eeg_matrix = eeg_df.pivot(index="time", columns="epoch", values="F7")
    time_points = eeg_matrix.index.to_numpy(dtype=float)
    data_matrix = eeg_matrix.to_numpy()  # shape: (n_timepoints, n_epochs)

    # 3. Create FDataGrid (one curve per epoch)
    fd_grid = FDataGrid(
        data_matrix=data_matrix.T,  # skfda expects (n_samples, n_timepoints)
        grid_points=time_points
    )

    # 4. Define B-spline basis (nbasis=20, matching R)
    time_range = (time_points[0], time_points[-1])
    basis = BSplineBasis(domain_range=time_range, n_basis=20)

    # 5. Smooth into functional data basis representation
    smoother = BasisSmoother(basis)
    eeg_fd = smoother.fit_transform(fd_grid)

    # 3. Construct dynamic output path
    new_path = str(path).replace(input_root, output_root)
    target_dir = Path(new_path).parent
    file_name_clean = path.stem  # filename without .csv

    # 4. Create output folder if missing
    target_dir.mkdir(parents=True, exist_ok=True)

    # 5. Save as .pkl
    final_pkl_path = target_dir / f"{file_name_clean}.pkl"
    with open(final_pkl_path, "wb") as f:
        pickle.dump(eeg_fd, f)

    print(f"Processed: {file_name_clean}")
    print(f"Saved to:  {final_pkl_path}")
    print("-----------------------------------")
    
# ── Timer end ─────────────────────────────────────────────────────────────────────────
print("───────────────────────────────────────────────────────────────────────────────")
print(f"  Time elapsed: {format_elapsed(time.time() - start)}")
print("───────────────────────────────────────────────────────────────────────────────")
# ──────────────────────────────────────────────────────────────────────────────────────

Processed: task-auditoryoddball_Stimulus_S1
Saved to:  ds006018_functional/sub-001/task-auditoryoddball_Stimulus_S1.pkl
-----------------------------------
Processed: task-auditoryoddball_Stimulus_S180
Saved to:  ds006018_functional/sub-001/task-auditoryoddball_Stimulus_S180.pkl
-----------------------------------
Processed: task-auditoryoddball_Stimulus_S70
Saved to:  ds006018_functional/sub-001/task-auditoryoddball_Stimulus_S70.pkl
-----------------------------------
Processed: task-auditoryoddball_Stimulus_S80
Saved to:  ds006018_functional/sub-001/task-auditoryoddball_Stimulus_S80.pkl
-----------------------------------
Processed: task-flanker_Stimulus_S11
Saved to:  ds006018_functional/sub-001/task-flanker_Stimulus_S11.pkl
-----------------------------------
Processed: task-flanker_Stimulus_S111
Saved to:  ds006018_functional/sub-001/task-flanker_Stimulus_S111.pkl
-----------------------------------
Processed: task-flanker_Stimulus_S112
Saved to:  ds006018_functional/sub-001/task-